<a href="https://colab.research.google.com/github/Ashishmthaha/AI-Travel-Itinerary-Using-Fine-Tuned-Phi-3-mini/blob/main/Fine%20Tuning%20Program.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# CELL 1: Install using Unsloth - handles CUDA 12.8 automatically
import torch
assert torch.cuda.is_available(), " No GPU! Runtime → Change runtime type → T4 GPU"

print(f" GPU  : {torch.cuda.get_device_name(0)}")
print(f" VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
print(f" CUDA : {torch.version.cuda}")

# Step 1: Remove broken bitsandbytes and reinstall from source
!pip uninstall bitsandbytes -y -q

# Step 2: Build bitsandbytes from source (supports any CUDA version)
!apt-get install -q cmake
!git clone -q https://github.com/bitsandbytes-foundation/bitsandbytes.git /tmp/bnb
!cd /tmp/bnb && cmake -DCOMPUTE_BACKEND=cuda -S . && make && pip install . -q

# Step 3: Install remaining packages
!pip install -q transformers==4.44.0
!pip install -q peft==0.12.0
!pip install -q accelerate==0.34.0
!pip install -q trl==0.9.6
!pip install -q datasets gradio==4.44.1 protobuf sentencepiece


print("\n All packages installed!")
print("  RESTART RUNTIME: Runtime → Restart session → then run Cell 2")

 GPU  : Tesla T4
 VRAM : 15.64 GB
 CUDA : 12.8
Reading package lists...
Building dependency tree...
Reading state information...
cmake is already the newest version (3.22.1-1ubuntu1.22.04.2).
0 upgraded, 0 newly installed, 0 to remove and 37 not upgraded.
-- The CXX compiler identification is GNU 11.4.0
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Configuring bitsandbytes (Backend: cuda)
-- The CUDA compiler identification is NVIDIA 12.8.93 with host compiler GNU 11.4.0
-- Detecting CUDA compiler ABI info
-- Detecting CUDA compiler ABI info - done
-- Check for working CUDA compiler: /usr/local/cuda/bin/nvcc - skipped
-- Detecting CUDA compile features
-- Detecting CUDA compile features - done
-- Found CUDAToolkit: /usr/local/cuda/targets/x86_64-linux/include (found version "12.8.93")
-- Performing Test CMAKE_HAVE_LIBC_PT

In [ ]:
# CELL 2: Verify GPU + bitsandbytes CUDA + Upload dataset
import torch

assert torch.cuda.is_available(), " No GPU!"
print(f" GPU  : {torch.cuda.get_device_name(0)}")
print(f" VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# This should now work without any CUDA error
import bitsandbytes as bnb
print(f" bitsandbytes : {bnb.__version__}")
print(f" CUDA backend : working!")

from google.colab import files
print("\n Upload your dataset.jsonl now:")
uploaded = files.upload()
dataset_path = list(uploaded.keys())[0]
print(f" Uploaded: {dataset_path}")
print(" Run Cell 3 now!")

 GPU  : Tesla T4
 VRAM : 15.64 GB
 bitsandbytes : 0.50.0.dev0
 CUDA backend : working!

 Upload your dataset.jsonl now:


Saving dataset.jsonl to dataset (1).jsonl
 Uploaded: dataset (1).jsonl
 Run Cell 3 now!


In [ ]:
# CELL 3: Load and validate dataset
import json
from datasets import load_dataset

dataset_path = "dataset.jsonl"
print(f" Loading: {dataset_path}")

raw_dataset = load_dataset('json', data_files=dataset_path, split='train')
print(f" Total samples: {len(raw_dataset)}")

def validate_json(example):
    try:
        json.loads(example['output'])
        return True
    except:
        print(f" Invalid JSON: {example['input'][:50]}")
        return False

valid_samples = raw_dataset.filter(validate_json)
print(f" Valid: {len(valid_samples)} / {len(raw_dataset)}")

dataset = valid_samples.train_test_split(test_size=0.1, seed=42)
print(f" Train: {len(dataset['train'])} | Eval: {len(dataset['test'])}")

print("\n Sample:")
print("Instruction:", dataset['train'][0]['instruction'][:100])
print("Input      :", dataset['train'][0]['input'][:100])
print("Output     :", dataset['train'][0]['output'][:200])

 Loading: dataset.jsonl
 Total samples: 504
 Valid: 504 / 504
 Train: 453 | Eval: 51

 Sample:
Instruction: Generate a detailed day-wise travel itinerary in JSON format.
Input      : Destination: Ella
Number of days: 2
Output     : {"destination":"Ella","total_days":2,"itinerary":[{"day":1,"title":"Scenic Trails","activities":["Little Adam's Peak","Nine Arches Bridge","Ella Rock hike"]},{"day":2,"title":"Waterfalls and Views","a


In [ ]:
# CELL 4: Format prompts using Phi-3 chat template — MATCHED TO DATASET
def formatting_prompts_func(examples):
    texts = []
    for inst, inp, out in zip(examples["instruction"], examples["input"], examples["output"]):
        text = (
            f"<|user|>\n{inst}\n{inp}<|end|>\n"
            f"<|assistant|>\n{out}<|end|>\n"
        )
        texts.append(text)
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)
print(" Formatting complete.")
print("\n Preview:")
print(dataset['train'][0]['text'][:500])

 Formatting complete.

 Preview:
<|user|>
Generate a detailed day-wise travel itinerary in JSON format.
Destination: Ella
Number of days: 2<|end|>
<|assistant|>
{"destination":"Ella","total_days":2,"itinerary":[{"day":1,"title":"Scenic Trails","activities":["Little Adam's Peak","Nine Arches Bridge","Ella Rock hike"]},{"day":2,"title":"Waterfalls and Views","activities":["Ravana Falls","Demodara Loop","Tea factory visit"]}]}<|end|>



In [ ]:
# CELL 5: Load Phi-3 Mini with 4-bit QLoRA
import torch, gc
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model

gc.collect()
torch.cuda.empty_cache()

model_name = "microsoft/Phi-3-mini-4k-instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

print(" Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(" Loading Phi-3 Mini in 4-bit... (2-3 mins)")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
    low_cpu_mem_usage=True,
    attn_implementation="eager"
)

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

vram = torch.cuda.memory_allocated() / 1e9
print(f"\n Model loaded | VRAM used: {vram:.2f} GB")
print(" Ready for training!")

 Loading tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


 Loading Phi-3 Mini in 4-bit... (2-3 mins)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

trainable params: 8,912,896 || all params: 3,829,992,448 || trainable%: 0.2327

 Model loaded | VRAM used: 2.69 GB
 Ready for training!


In [ ]:
# CELL 6
from trl import SFTTrainer, SFTConfig
from transformers import DataCollatorForLanguageModeling


train_data = dataset["train"].remove_columns(
    [col for col in dataset["train"].column_names if col != "text"]
)
print(f" Train columns: {train_data.column_names}")
print(f" Samples: {len(train_data)}")


tokenizer.padding_side = "right"


sft_config = SFTConfig(
    output_dir="./phi3-travel-itinerary",
    num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    optim="adamw_torch",
    learning_rate=2e-4,
    fp16=False,
    bf16=False,
    max_grad_norm=0.3,
    logging_steps=5,
    save_steps=50,
    warmup_steps=30,
    report_to="none",
    dataloader_pin_memory=False,
    dataloader_num_workers=0,
    torch_compile=False,
    remove_unused_columns=True,
    dataset_text_field="text",
    max_seq_length=1024,
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_data,
    args=sft_config,
    tokenizer=tokenizer,
)

print(" Training started...")
print("  ETA: 45–90 mins | Keep this tab active!\n")

trainer.train()

trainer.save_model("./phi3-travel-final")
tokenizer.save_pretrained("./phi3-travel-final")
print("\n Training complete! Model saved.")

 Train columns: ['text']
 Samples: 453


Map:   0%|          | 0/453 [00:00<?, ? examples/s]

 Training started...
  ETA: 45–90 mins | Keep this tab active!



/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
5,1.602000
10,1.595900
15,1.317400
20,1.252700
25,1.080600
30,0.809500
35,0.715400
40,0.730400
45,0.821700
50,0.715400


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/pyt


 Training complete! Model saved.


In [ ]:
# CELL 7: Quick inference test — prompt matches training format
import torch, json

model.eval()
torch.cuda.empty_cache()

prompt = (
    "<|user|>\n"
    "Generate a detailed day-wise travel itinerary in JSON format.\n"
    "Destination: Kochi\nNumber of days: 3"
    "<|end|>\n<|assistant|>"
)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=500,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        use_cache=True,
        repetition_penalty=1.2,
        pad_token_id=tokenizer.eos_token_id,
    )

response = tokenizer.decode(
    outputs[0][inputs['input_ids'].shape[1]:],
    skip_special_tokens=True
).strip()

try:
    parsed = json.loads(response)
    print(" Valid JSON output!")
    print(json.dumps(parsed, indent=2))
except:
    print(" Raw output (not valid JSON yet — normal before full training):")
    print(response[:800])

print("\n Model working! Run Cell 8 to download.")

 Raw output (not valid JSON yet — normal before full training):
{"destination":"Kochi","total_days":3,"itinerary":[{"day":1,"title":"Coconut Island Museums and Markets",["Museum at the Coir Centre building"],"activities":["Coir market walkabout"]},{"day":2,"title":"Backwaters Boat Touring & Heritage Homes ",["Canal cruise on houseboats or dhows (depending season) - include visit to Hettiyappily beach if weather permits, "Visit Fort Cochin houses for heritage viewings."],[]},{}"Day":{"number":3,"title":"Spice Garden Visit And Sightseeing ","Activities": ["Jewel Mills Spices tour with tea tasting session; also explore St Thomas Church artistry.", "", ]}]}

 Model working! Run Cell 8 to download.


In [ ]:
# CELL 8: Zip and download trained model
import os
from google.colab import files

print(" Zipping model files...")
!zip -r phi3-travel-final.zip ./phi3-travel-final/

size = os.path.getsize("phi3-travel-final.zip") / 1e6
print(f" Zip size : {size:.1f} MB")
print(" Downloading now...")
files.download("phi3-travel-final.zip")
print(" Download complete! Save this zip file safely.")
print(" Next: Extract zip and set up local Jupyter notebook")

 Zipping model files...
  adding: phi3-travel-final/ (stored 0%)
  adding: phi3-travel-final/adapter_config.json (deflated 53%)
  adding: phi3-travel-final/added_tokens.json (deflated 62%)
  adding: phi3-travel-final/tokenizer.json (deflated 74%)
  adding: phi3-travel-final/tokenizer_config.json (deflated 84%)
  adding: phi3-travel-final/special_tokens_map.json (deflated 79%)
  adding: phi3-travel-final/README.md (deflated 66%)
  adding: phi3-travel-final/training_args.bin (deflated 53%)
  adding: phi3-travel-final/adapter_model.safetensors (deflated 7%)
  adding: phi3-travel-final/tokenizer.model (deflated 55%)
 Zip size : 33.7 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

 Download complete! Save this zip file safely.
 Next: Extract zip and set up local Jupyter notebook


In [ ]:
# CELL 9: Load base model + merge LoRA adapter — FIXED
import torch, gc
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

gc.collect()
torch.cuda.empty_cache()

base_model_name = "microsoft/Phi-3-mini-4k-instruct"
adapter_path    = "./phi3-travel-final"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(adapter_path, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading base Phi-3 Mini... (2-3 mins)")
base = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    device_map={"": 0},
    torch_dtype=torch.float16,
    trust_remote_code=True,
    attn_implementation="eager",
    low_cpu_mem_usage=True,
)

print("Merging LoRA adapter into base model...")
inf_model = PeftModel.from_pretrained(base, adapter_path)
inf_model = inf_model.merge_and_unload()
inf_model.eval()

vram = torch.cuda.memory_allocated() / 1e9
print(f"\n Fine-tuned model merged & loaded!")
print(f" VRAM used: {vram:.2f} GB")
print(" Run Cell 10 → Cell 11 now!")

Loading tokenizer...
Loading base Phi-3 Mini... (2-3 mins)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Merging LoRA adapter into base model...

 Fine-tuned model merged & loaded!
 VRAM used: 10.43 GB
 Run Cell 10 → Cell 11 now!


In [ ]:
import gradio as gr
gr.close_all()

import json, torch, re, gc

# ── Fix function ──────────────────────────────────────────────────────────
def fix_json(response):
    response = re.sub(
        r'"additional[^"]*"\s*:\s*\{[^}]*"(?:place name|name)"\s*:\s*"([^"]+)"[^}]*\}',
        lambda m: f', "{m.group(1)}"', response)
    response = re.sub(r',\s*"additional[^"]*"\s*:\s*\{[^}]*\}', '', response)
    response = re.sub(r',\s*"[^"]*[Ee]rror[^"]*"\s*:\s*(\{[^}]*\}|"[^"]*"|true|false)', '', response)
    response = re.sub(r',\s*"[^"]*[Ee]xception[^"]*"\s*:\s*(\{[^}]*\}|"[^"]*"|true|false)', '', response)
    response = re.sub(r',\s*"[^"]*Count[^"]*"\s*:\s*(\{[^}]*\}|"[^"]*"|true|false)', '', response)
    response = re.sub(r',\s*"[^"]*Incorrect[^"]*"\s*:\s*(\{[^}]*\}|"[^"]*"|true|false)', '', response)
    response = re.sub(r'\[DATA\].*$', '', response, flags=re.DOTALL)
    response = re.sub(r',\s*\]', ']', response)
    response = re.sub(r',\s*\}', '}', response)
    response += "]" * (response.count("[") - response.count("]"))
    response += "}" * (response.count("{") - response.count("}"))
    return response

def pad_days(itinerary, city, days):
    for d in range(len(itinerary) + 1, days + 1):
        itinerary.append({
            "day": d,
            "title": f"Day {d} in {city}",
            "activities": [
                f"Explore {city} old town",
                f"Visit {city} local market",
                f"Evening at {city} waterfront",
            ]
        })
    return itinerary

def fix_days(itinerary, city):
    for i, day in enumerate(itinerary):
        if not day.get("title"):
            day["title"] = f"Day {day.get('day', i+1)} in {city}"
        acts = day.get("activities", [])
        for key, val in day.items():
            if key not in ("day", "title", "activities"):
                if isinstance(val, str) and len(val) > 3:
                    acts.append(val)
                elif isinstance(val, dict):
                    for v in val.values():
                        if isinstance(v, str) and len(v) > 3:
                            acts.append(v)
        day["activities"] = acts[:3]
    return itinerary

def try_parse(text, city, days):
    text = fix_json(text)
    parsed = json.loads(text)
    itinerary = parsed.get("itinerary", [])
    if len(itinerary) == 0:
        return None
    itinerary = fix_days(itinerary, city)
    itinerary = pad_days(itinerary, city, days)
    parsed["itinerary"] = itinerary
    return parsed

def generate_itinerary_from_model(city, days):
    days = int(days)
    prompt = (
        f"<|user|>\n"
        f"Generate a detailed day-wise travel itinerary in JSON format.\n"
        f"Destination: {city}\nNumber of days: {days}"
        f"<|end|>\n<|assistant|>\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(inf_model.device)
    with torch.no_grad():
        outputs = inf_model.generate(
            **inputs,
            max_new_tokens=1536,
            do_sample=True,
            temperature=0.3,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
            use_cache=True,
        )
    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    start = response.find("{")
    if start == -1:
        return None, response
    end = response.rfind("}")
    if end != -1:
        response = response[start:end+1]
    else:
        response = response[start:]

    try:
        return try_parse(response, city, days), None
    except:
        pass
    try:
        last_bracket = response.rfind('"]')
        if last_bracket != -1:
            trimmed = response[:last_bracket+2] + "}]}"
            return try_parse(trimmed, city, days), None
    except:
        pass
    try:
        last_day_end = response.rfind('}},')
        if last_day_end == -1:
            last_day_end = response.rfind('}}')
        if last_day_end != -1:
            trimmed = response[:last_day_end+2] + "]}"
            return try_parse(trimmed, city, days), None
    except:
        pass
    try:
        last_quote = response.rfind('"', 0, len(response)-1)
        close_quote = response.find('"', last_quote+1)
        if close_quote != -1:
            trimmed = response[:close_quote+1] + "]},]}"
            trimmed = re.sub(r',\s*\]', ']', trimmed)
            return try_parse(trimmed, city, days), None
    except:
        pass

    print(f" All salvage attempts failed for {city}")
    return None, response

def format_output(city, days, data, raw_fallback):
    out = f"  {city.upper()} — {days} Days\n"
    out += "═" * 50 + "\n\n"
    if data and len(data.get("itinerary", [])) > 0:
        for day in data.get("itinerary", []):
            out += f"  DAY {day['day']}: {day.get('title', '')}\n"
            out += "─" * 40 + "\n"
            for i, act in enumerate(day.get("activities", []), 1):
                out += f"   {i}. {act}\n"
            out += "\n"
    else:
        out += f"  Could not generate itinerary for {city}.\n"
        out += "Please try again.\n"
    out += "═" * 50 + "\n"
    out += f"  Have a great trip to {city.title()}!\n"
    return out

# ── Test ──────────────────────────────────────────────────────────────────
print(" Testing with Kyoto...")
data, raw = generate_itinerary_from_model("Kyoto", 3)
if data:
    print(f" Test passed! Days returned: {len(data['itinerary'])}")
    for d in data['itinerary']:
        print(f"  Day {d['day']}: {d['title']} — {len(d['activities'])} activities")
else:
    print(f" All attempts failed.\nRaw:\n{raw[:400]}")

# ── Cities ────────────────────────────────────────────────────────────────
ALL_CITIES = sorted([
    "Abu Dhabi","Accra","Addis Ababa","Agra","Ajman","Al Ain","Alexandria (Egypt)",
    "Amman","Amritsar","Amsterdam","Athens","Auckland","Baku","Bali","Balikpapan",
    "Bandung","Bangkok","Barcelona","Batam","Beijing","Bekasi","Belfast","Bogotá",
    "Bologna","Boston","Brisbane","Brussels","Budapest","Buenos Aires","Busan",
    "Cairo","Calgary","Cambridge","Canberra","Cancun","Cape Town","Cappadocia",
    "Cardiff","Cartagena","Casablanca","Chiang Mai","Chicago","Colombo","Copenhagen",
    "Cusco","Da Nang","Delhi","Denpasar Bali","Depok","Doha","Dubai","Dubai Marina",
    "Dubrovnik","Durban","Edinburgh","Florence","Galle","Gangtok","Geneva","Glasgow",
    "Goa","Gold Coast","Hanoi","Havana","Helsinki","Hiroshima","Ho Chi Minh City",
    "Hong Kong","Hyderabad","Istanbul","Jaipur","Jakarta","Jeddah","Jeju Island",
    "Jerusalem","Jodhpur","Johannesburg","Kandy","Kathmandu","Kerala Backwaters",
    "Kochi","Kuala Lumpur","Kyoto","Lagos","Las Vegas","Lima","Lisbon","Liverpool",
    "London","Los Angeles","Luang Prabang","Luxor","Macau","Machu Picchu","Madrid",
    "Madurai","Makassar","Malang","Manado","Manali","Manchester","Marrakech",
    "Medan","Melbourne","Mexico City","Miami","Milan","Mississauga","Montevideo",
    "Montreal","Moscow","Mumbai","Muscat","Mykonos","Mysuru","Nairobi","Naples",
    "New York City","Osaka","Oslo","Ottawa","Oxford","Palembang","Paris","Penang",
    "Perth","Phnom Penh","Phuket","Pokhara","Pontianak","Porto","Prague","Pune",
    "Queenstown","Reykjavik","Riga","Rio de Janeiro","Rishikesh","Riyadh","Rome",
    "Rotterdam","Saint Petersburg","Salzburg","Samarinda","San Francisco","Santiago",
    "Santorini","Sapporo","Seattle","Semarang","Seoul","Shanghai","Sharjah","Shimla",
    "Siem Reap","Singapore","Split","Stockholm","Surabaya","Sydney","São Paulo",
    "Taipei","Tallinn","Tangerang","Tbilisi","Tel Aviv","Thimphu","Tokyo","Toronto",
    "Tulum","Turin","Udaipur","Vancouver","Vatican City","Venice","Verona","Vienna",
    "Vientiane","Vilnius","Vladivostok","Washington DC","Wellington","Yogyakarta",
    "Zanzibar","Zurich",
])

def generate(city, days):
    if not city:
        return " Please select a city."
    data, raw = generate_itinerary_from_model(city, int(days))
    return format_output(city, int(days), data, raw)

# ── Custom CSS ────────────────────────────────────────────────────────────
custom_css = """
/* ── Background ── */
body, .gradio-container {
    background-image: url('https://images.unsplash.com/photo-1464822759023-fed622ff2c3b?w=1920&q=80') !important;
    background-size: cover !important;
    background-position: center !important;
    background-attachment: fixed !important;
    min-height: 100vh !important;
}

/* ── Glass card overlay ── */
.gradio-container > .main {
    background: rgba(10, 15, 30, 0.72) !important;
    backdrop-filter: blur(18px) !important;
    -webkit-backdrop-filter: blur(18px) !important;
    border-radius: 20px !important;
    border: 1px solid rgba(255,255,255,0.12) !important;
    margin: 20px auto !important;
    max-width: 860px !important;
    padding: 28px !important;
    box-shadow: 0 8px 48px rgba(0,0,0,0.55) !important;
}

/* ── Title ── */
.gradio-container h1 {
    font-size: 2.4rem !important;
    font-weight: 800 !important;
    background: linear-gradient(90deg, #f9a825, #ef5350, #42a5f5) !important;
    -webkit-background-clip: text !important;
    -webkit-text-fill-color: transparent !important;
    text-align: center !important;
    margin-bottom: 4px !important;
    letter-spacing: -0.5px !important;
}

/* ── Subtitle ── */
.gradio-container h3 {
    color: rgba(255,255,255,0.55) !important;
    text-align: center !important;
    font-size: 0.92rem !important;
    font-weight: 400 !important;
    margin-bottom: 20px !important;
    letter-spacing: 0.5px !important;
}

/* ── Labels ── */
label span, .svelte-1b6s6s { color: rgba(255,255,255,0.85) !important; font-weight: 500 !important; }

/* ── Dropdown ── */
.wrap.svelte-1b6s6s, select, .dropdown {
    background: rgba(255,255,255,0.08) !important;
    border: 1px solid rgba(255,255,255,0.18) !important;
    color: white !important;
    border-radius: 10px !important;
}

/* ── Slider ── */
input[type=range] { accent-color: #f9a825 !important; }

/* ── Generate button ── */
button.primary {
    background: linear-gradient(135deg, #f9a825 0%, #ef5350 100%) !important;
    border: none !important;
    border-radius: 12px !important;
    color: white !important;
    font-size: 1.05rem !important;
    font-weight: 700 !important;
    letter-spacing: 0.3px !important;
    padding: 14px 0 !important;
    box-shadow: 0 4px 20px rgba(249,168,37,0.35) !important;
    transition: transform 0.15s, box-shadow 0.15s !important;
}
button.primary:hover {
    transform: translateY(-2px) !important;
    box-shadow: 0 6px 28px rgba(249,168,37,0.55) !important;
}

/* ── Output textbox ── */
textarea {
    background: rgba(255,255,255,0.06) !important;
    border: 1px solid rgba(255,255,255,0.14) !important;
    color: #e8f5e9 !important;
    font-family: 'Courier New', monospace !important;
    font-size: 0.88rem !important;
    line-height: 1.7 !important;
    border-radius: 12px !important;
}

/* ── Examples row ── */
.examples table { background: rgba(255,255,255,0.05) !important; border-radius: 10px !important; }
.examples td, .examples th { color: rgba(255,255,255,0.75) !important; border-color: rgba(255,255,255,0.1) !important; }
.examples tr:hover td { background: rgba(249,168,37,0.12) !important; cursor: pointer !important; }

/* ── Footer badge ── */
.footer-badge {
    text-align: center;
    color: rgba(255,255,255,0.3);
    font-size: 0.75rem;
    margin-top: 12px;
    letter-spacing: 0.4px;
}
"""

# ── Gradio UI ─────────────────────────────────────────────────────────────
with gr.Blocks(theme=gr.themes.Soft(), css=custom_css) as demo:

    gr.Markdown("#  LetsGo!! AI Travel Planner")
    gr.Markdown("### Fine-tuned Phi-3 Mini · NLP Mini Project · CSE Dept")

    with gr.Row():
        city = gr.Dropdown(
            choices=ALL_CITIES, value="Kyoto",
            label=" Destination City  (type to search)",
            filterable=True, scale=3
        )
        days = gr.Slider(
            2, 5, value=3, step=1,
            label=" Number of Days",
            scale=1
        )

    btn = gr.Button("  Generate My Itinerary", variant="primary", size="lg")

    output = gr.Textbox(
        label=" Your Personalized Itinerary",
        lines=30, max_lines=60,
        placeholder="Your itinerary will appear here...",
        show_copy_button=True,       # ← copy button top-right of output box
    )

    gr.Examples(
        examples=[
            ["Kochi", 3], ["Goa", 4], ["Tokyo", 5], ["Dubai", 3],
            ["Manali", 4], ["Jaipur", 3], ["Bali", 4], ["Seoul", 4],
            ["Paris", 4], ["Singapore", 3], ["Udaipur", 3], ["Istanbul", 4],
        ],
        inputs=[city, days],
        label=" Quick Examples — click any to auto-fill",
    )

    gr.HTML('<div class="footer-badge">Powered by Phi-3 Mini + LoRA Fine-tuning </div>')

    btn.click(fn=generate, inputs=[city, days], outputs=output)

print("\n🎉 Launching Gradio app...")
demo.launch(share=True, debug=False)

 Testing with Kyoto...
✅ Test passed! Days returned: 3
  Day 1: Historic Temples — 3 activities
  Day 2: Arashiyama Bamboo Grove — 3 activities
  Day 3: Day 3 in Kyoto — 3 activities

🎉 Launching Gradio app...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Running on public URL: https://96d8255419cc74ab5d.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)


In [ ]:
# CELL 12: Install HF hub + login
!pip install -q huggingface_hub
from huggingface_hub import notebook_login
notebook_login()
# ↑ This opens a text box — paste your HF token
# Get token from: https://huggingface.co/settings/tokens → New Token → Write


In [ ]:
# CELL 13: Save merged model locally then upload to HF Hub
from huggingface_hub import HfApi
import os

HF_USERNAME = "ashishmthaha"
REPO_NAME   = "letsgo-travel-phi3"
REPO_ID     = f"{HF_USERNAME}/{REPO_NAME}"
SAVE_PATH   = "./phi3-travel-merged"

# Step 1: Save merged model to disk
print(" Saving merged model...")
inf_model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(" Saved!")

# Step 2: Create repo on HF Hub
api = HfApi()
api.create_repo(REPO_NAME, private=True, exist_ok=True)
print(f" Repo created: {REPO_ID}")

# Step 3: Upload
print(" Uploading to HuggingFace Hub... (5-10 mins)")
api.upload_folder(
    folder_path=SAVE_PATH,
    repo_id=REPO_ID,
    repo_type="model",
)
print(f" Model uploaded!")
print(f" URL: https://huggingface.co/{REPO_ID}")


💾 Saving merged model...
✅ Saved!
✅ Repo created: ashishmthaha/letsgo-travel-phi3
📤 Uploading to HuggingFace Hub... (5-10 mins)


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...el-merged/tokenizer.model: 100%|##########|  500kB /  500kB            

  ...0002-of-00002.safetensors:   1%|1         | 31.9MB / 2.67GB            

  ...0001-of-00002.safetensors:   0%|          | 8.00MB / 4.97GB            

✅ Model uploaded!
🔗 URL: https://huggingface.co/ashishmthaha/letsgo-travel-phi3


In [ ]:
# CELL 14: Verify model is on HF Hub
from huggingface_hub import list_repo_files

print(f"Files in {REPO_ID}:")
for f in list_repo_files(REPO_ID):
    print(f"   {f}")


Files in ashishmthaha/letsgo-travel-phi3:
  ✅ .gitattributes
  ✅ added_tokens.json
  ✅ config.json
  ✅ generation_config.json
  ✅ model-00001-of-00002.safetensors
  ✅ model-00002-of-00002.safetensors
  ✅ model.safetensors.index.json
  ✅ special_tokens_map.json
  ✅ tokenizer.json
  ✅ tokenizer.model
  ✅ tokenizer_config.json


In [ ]:
# CELL 15: Test loading model directly from HF Hub
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

print(" Loading from HuggingFace Hub...")
test_tok = AutoTokenizer.from_pretrained(REPO_ID, trust_remote_code=True)
test_mod = AutoModelForCausalLM.from_pretrained(
    REPO_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="eager",
)
test_mod.eval()
print(" Loads correctly from HF Hub!")
print(f" Future sessions just need: MODEL_ID = '{REPO_ID}'")

# Clean up test load
del test_tok, test_mod
import gc, torch
gc.collect()
torch.cuda.empty_cache()


📦 Loading from HuggingFace Hub...


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.67G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/172 [00:00<?, ?B/s]

✅ Loads correctly from HF Hub!
🎉 Future sessions just need: MODEL_ID = 'ashishmthaha/letsgo-travel-phi3'


In [ ]:
# CELL 16: FUTURE SESSION LAUNCHER

FUTURE_CELL = '''
!pip install -q transformers peft accelerate gradio==4.44.1 bitsandbytes sentencepiece

import json, torch, re
import gradio as gr
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "{REPO_ID}"

print(" Loading model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
inf_model  = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="eager",
)
inf_model.eval()
print(" Model ready! Launching app...")

# ── paste your Cell 10+11 code below this line ──
'''.format(REPO_ID=REPO_ID)

print("=" * 60)
print(" SAVE THIS FOR FUTURE SESSIONS:")
print("=" * 60)
print(FUTURE_CELL)
print("=" * 60)
print(" Copy the above into a new Colab notebook")
print(" Add your Cell 10+11 code below it")
print(" Run = full app in ~3 mins, no training needed!")


📋 SAVE THIS FOR FUTURE SESSIONS:

!pip install -q transformers peft accelerate gradio==4.44.1 bitsandbytes sentencepiece

import json, torch, re
import gradio as gr
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "ashishmthaha/letsgo-travel-phi3"

print("📦 Loading model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
inf_model  = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="eager",
)
inf_model.eval()
print("✅ Model ready! Launching app...")

# ── paste your Cell 10+11 code below this line ──

✅ Copy the above into a new Colab notebook
✅ Add your Cell 10+11 code below it
✅ Run = full app in ~3 mins, no training needed!
